In [5]:
import numpy as np
import pandas as pd
import random
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from sentence_transformers import InputExample
from head_tail import  extract_head_tail

In [6]:
def train_cross_encoder(train_csv_path, output_model_path):
    print("Loading training data...")
    df = pd.read_csv(train_csv_path)
    texts = df['text'].tolist()
    
    # 1. Extract Edges
    heads = []
    tails = []
    for text in texts:
        h, t = extract_head_tail(text) # Assuming extract_head_tail is defined as above
        heads.append(h)
        tails.append(t)
        
    # 2. Generate Explicit Positive and Negative Pairs
    train_examples = []
    num_pages = len(texts)
    
    for i in range(num_pages - 1):
        # POSITIVE PAIR (Label 1.0)
        train_examples.append(InputExample(texts=[tails[i], heads[i+1]], label=1.0))
        
        # NEGATIVE PAIRS (Label 0.0)
        # For every positive pair, we generate 3 random negative pairs to balance the dataset
        for _ in range(3):
            random_idx = random.randint(0, num_pages - 1)
            # Make sure we don't accidentally pick the actual next page
            if random_idx != i + 1:
                train_examples.append(InputExample(texts=[tails[i], heads[random_idx]], label=0.0))
                
    random.shuffle(train_examples)
    print(f"Generated {len(train_examples)} total pairs for Cross-Encoder.")
    
    # Split a tiny portion off for evaluation during training
    train_split = int(len(train_examples) * 0.9)
    train_data = train_examples[:train_split]
    eval_data = train_examples[train_split:]
    
    # 3. Load Base Model (DeBERTa-v3 is excellent for NLI and sequence classification)
    # Using 'small' to fit in memory, but 'base' or 'large' will yield higher Kaggle scores
    print("Loading DeBERTa model...")
    model = CrossEncoder('microsoft/deberta-v3-small', num_labels=1)
    
    # 4. Setup Dataloader and Evaluator
    train_dataloader = DataLoader(train_data, shuffle=True, batch_size=8)
    evaluator = CEBinaryClassificationEvaluator.from_input_examples(eval_data, name='train-eval')
    
    # 5. Train Model
    print("Starting Cross-Encoder training...")
    model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=3,
        warmup_steps=100,
        output_path=output_model_path,
        show_progress_bar=True
    )
    print(f"Fine-tuned Cross-Encoder saved to {output_model_path}")

In [ ]:
train_cross_encoder('./data/Mysterious_Affair_at_Styles_Train_Data.csv', './models/fine_tuned_cross_encoder')

In [7]:
def solve_narrative_order_ce(input_csv, output_csv, model_path='./models/fine_tuned_cross_encoder'):
    print(f"Processing {input_csv}...")
    
    # Load dataset
    df = pd.read_csv(input_csv)
    page_ids = df['page'].tolist()
    texts = df['text'].tolist()
    num_pages = len(page_ids)
    
    # Apply Edge Extraction to all pages
    print("1. Extracting 60-word edges (Heads and Tails)...")
    heads, tails = zip(*[extract_head_tail(t) for t in texts])
    
    # ==========================================
    # STEP 2 & 3: Cross-Encoder Prediction
    # ==========================================
    print(f"2. Loading custom Cross-Encoder from '{model_path}'...")
    try:
        model = CrossEncoder(model_path)
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Make sure you have run the training script first and the path is correct.")
        return

    print("3. Building N x N Transition Matrix (This may take a moment)...")
    transition_matrix = np.zeros((num_pages, num_pages))
    
    for i in range(num_pages):
        # Create pairs of (Tail i, Head j) for all j
        pairs = [[tails[i], heads[j]] for j in range(num_pages)]
        
        # The model outputs a probability/logit score for all pairs at once
        scores = model.predict(pairs)
        transition_matrix[i] = scores
        
        # Optional: Print progress for large books
        if (i + 1) % 20 == 0:
            print(f"   Scored transitions for {i + 1}/{num_pages} pages...")
            
    # Prevent a page from linking to itself by setting the diagonal to -infinity
    np.fill_diagonal(transition_matrix, -np.inf)
    
    # ==========================================
    # STEP 4: Greedy Traversal
    # ==========================================
    print("4. Executing Greedy Traversal...")
    
    # Find Page 1: The page that is least likely to come AFTER any other page
    # We look at the maximum score each page gets as a "Head"
    max_incoming_scores = np.max(transition_matrix, axis=0)
    current_page_idx = np.argmin(max_incoming_scores)
    
    ordered_indices = [current_page_idx]
    visited = set([current_page_idx])
    
    # Build the chain
    for _ in range(num_pages - 1):
        # Get transition scores from the current page's tail to all other heads
        scores_from_current = transition_matrix[current_page_idx].copy()
        
        # Mask out pages we've already placed in the sequence
        for v in visited:
            scores_from_current[v] = -np.inf
            
        # Pick the best match
        next_page_idx = np.argmax(scores_from_current)
        
        # Add to our sequence and update state
        ordered_indices.append(next_page_idx)
        visited.add(next_page_idx)
        current_page_idx = next_page_idx

    # ==========================================
    # Formatting and Saving Submission
    # ==========================================
    print(f"Saving results to {output_csv}...")
    submission_data = {
        'original_page': range(1, num_pages + 1), # 1-based indexing required by evaluation
        'shuffled_page': [page_ids[i] for i in ordered_indices]
    }
    
    pd.DataFrame(submission_data).to_csv(output_csv, index=False)
    print("Done!\n")

In [8]:
solve_narrative_order_ce('./data/BookA_test.csv', './submissions/BookA_cross_encoder.csv')
solve_narrative_order_ce('./data/BookB_test.csv', './submissions/BookB_cross_encoder.csv')

Processing ./data/BookA_test.csv...
1. Extracting 60-word edges (Heads and Tails)...
2. Loading custom Cross-Encoder from './models/fine_tuned_cross_encoder'...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 4175.00it/s]


3. Building N x N Transition Matrix (This may take a moment)...


KeyboardInterrupt: 